In [1]:
# Importing necessary libraries:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Load the uploaded NYC TRAIN dataset:
df = pd.read_parquet("../data/train_NYC_inspection.parquet")

# Display the first few rows:
df.head()

,CUISINE DESCRIPTION,BORO,ZIPCODE,Latitude,Longitude,INSPECTION DATE,BUILDING,STREET,SCORE,GRADE
0,Chinese,Queens,11354.0,40.759778,-73.829235,2024-08-05,136-20,ROOSEVELT AVENUE,9.0,A
1,Coffee/Tea,Queens,11362.0,40.769834,-73.736180,2025-10-31,25201,NORTHERN BLVD,9.0,A
2,Japanese,Manhattan,10036.0,40.759161,-73.990369,2025-05-02,354,WEST 44 STREET,14.0,B
3,Pizza,Queens,11372.0,40.756245,-73.878681,2025-04-15,8906,NORTHERN BLVD,40.0,C
4,Chicken,Queens,11432.0,40.708211,-73.803110,2024-02-29,87-44,PARSONS BOULEVARD,20.0,None


In [2]:
# Defining our target column"
target_column = 'GRADE'

# Handling missing values in the target column:
df = df.dropna(subset=[target_column])

# Separating target and features:
X = df.drop(columns=[target_column])
y = df[target_column]

# Preprocessing setup:

# Identifying numeric and categorical features:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Imputing missing values in features (if any):
# Must impute since SVM can not handle missing values directly.
num_imputer = SimpleImputer(strategy='median') # Using median for numeric features
X_num_imputed = num_imputer.fit_transform(X[numeric_features])

# Our separated features:
print(f"Numeric Features: {numeric_features}")
print(f"Categorical Features: {categorical_features}")

Numeric Features: Index(['ZIPCODE', 'Latitude', 'Longitude', 'SCORE'], dtype='object')
Categorical Features: Index(['CUISINE DESCRIPTION', 'BORO', 'BUILDING', 'STREET'], dtype='object')


For dealing with missing values, we have decided to impute them using the median for numerical columns and the mode for categorical columns. This approach helps to maintain the central tendency of the data without being overly influenced by outliers. We must deal with missing values specifically for SVM, as it does not handle them natively. 

In [3]:
# Must scale for SMV (since it is sensitive to feature scaling):
scaler = StandardScaler()
# SCaling numeric features:
X_num_scaled = scaler.fit_transform(X_num_imputed)

# CATEGORICAL features preprocessing:

# One-hot encoding categorical features:
cat_imputer = SimpleImputer(strategy='most_frequent') # Using most frequent for categorical features imputations
X_cat_imputed = cat_imputer.fit_transform(X[categorical_features])

# Encoding filled data:
cat_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_cat_encoded = cat_encoder.fit_transform(X_cat_imputed)

# Combining scaled numeric features and encoded categorical features:
X_final = np.hstack((X_num_scaled, X_cat_encoded))


In [4]:
# Using a random 15% of the data(for quicker optimization during demo):
X_tune, X_rest, y_tune, y_rest = train_test_split(
    X_final, y, train_size=0.15, random_state=42, stratify=y
)


In [5]:
# Model Optimization:
#svm = SVC(random_state=42)

#param_grid = {
#    'C': [0.1, 1, 10],
 #   'kernel': ['linear', 'rbf'],
  #  'gamma': ['scale', 'auto']
#}
#from sklearn.model_selection import GridSearchCV
#grid_search = GridSearchCV(svm, param_grid, cv=3, scoring='accuracy', n_jobs=-2, verbose=1)

#grid_search.fit(X_tune, y_tune)

In [6]:
from sklearn.svm import LinearSVC

# Using linear SVM for faster training on large datasets:
linear_svm = LinearSVC(dual=False, random_state=42)
linear_svm.fit(X_tune, y_tune) 

print("Score:", linear_svm.score(X_tune, y_tune))

Score: 0.9663840982168956


In [7]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV

# Using linear SVM for hyperparameter tuning:
linear_svm = LinearSVC(dual=False, random_state=42)

# We can still tune 'C' (Penalty)
param_grid = {
    'C': [0.01, 0.1, 1, 10]
}

grid_search = GridSearchCV(linear_svm, param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)

print("Starting LinearSVC training...")
grid_search.fit(X_final, y) # Using the entire dataset for tuning

print("Best Params:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Starting LinearSVC training...
Fitting 5 folds for each of 4 candidates, totalling 20 fits
Best Params: {'C': 10}
Best Score: 0.8977536187383969


Decided to use LinearSVC from sklearn for SVM implementation. LinearSVC is efficient for large datasets and works well with high-dimensional data, making it a suitable choice for our classification tasks. Although we lose the ability to graphically represent the SVM decision boundary in higher dimensions, the performance benefits outweigh this drawback.